# Healthcare Analytics: Exploratory Data Analysis (EDA)
This notebook explores the healthcare dataset to understand features, search for missing values, analyze outliers, detect class imbalance, and view feature correlations.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Load the Dataset


In [ ]:
df_path = os.path.join('..', 'data', 'healthcare_dataset.csv')
df = pd.read_csv(df_path)
print(f'Dataset Shape: {df.shape}')
df.head()


## 2. Basic Dataset Information & Summary Statistics


In [ ]:
df.info()


In [ ]:
df.describe().T


## 3. Missing Value Analysis


In [ ]:
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Percentage (%)': missing_percent})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)
print('Features with missing values:')
print(missing_df)

# Plot missing values
if not missing_df.empty:
    sns.barplot(x=missing_df.index, y='Missing Count', data=missing_df, palette='viridis')
    plt.title('Missing Value Count per Feature')
    plt.xlabel('Features')
    plt.ylabel('Count')
    plt.show()


## 4. Distribution of Target Variable: Stroke
Check class imbalance, which is common in healthcare prediction datasets.


In [ ]:
stroke_counts = df['Stroke'].value_counts()
stroke_pct = df['Stroke'].value_counts(normalize=True) * 100
print('Stroke Distribution:')
for val, count, pct in zip(stroke_counts.index, stroke_counts.values, stroke_pct.values):
    print(f'Class {val}: {count} records ({pct:.2f}%)')

plt.figure(figsize=(6, 5))
sns.countplot(x='Stroke', data=df, palette='pastel')
plt.title('Stroke Class Distribution')
plt.xlabel('Stroke (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()


## 5. Numerical Feature Distributions
Analyze the distribution of continuous clinical indicators: `Age`, `Avg_Glucose_Level`, `BMI`.


In [ ]:
num_cols = ['Age', 'Avg_Glucose_Level', 'BMI']
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
for i, col in enumerate(num_cols):
    # Histogram
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color='skyblue')
    axes[i, 0].set_title(f'{col} Histogram')
    
    # Boxplot for Outliers
    sns.boxplot(x=df[col], ax=axes[i, 1], color='lightgreen')
    axes[i, 1].set_title(f'{col} Boxplot (Outlier Detection)')

plt.tight_layout()
plt.show()


## 6. Categorical Feature Distributions
Explore distributions of demographic and administrative variables.


In [ ]:
cat_cols = ['Gender', 'Hypertension', 'Heart_Disease', 'Ever_Married', 'Work_Type', 'Residence_Type', 'Smoking_Status', 'Admission_Type', 'Medical_Condition', 'Insurance_Provider', 'Risk_Category']

fig, axes = plt.subplots(4, 3, figsize=(20, 24))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(x=col, data=df, ax=axes[i], palette='muted')
    axes[i].set_title(f'{col} Count')
    axes[i].tick_params(axis='x', rotation=45)
    
# Remove empty subplots
for j in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## 7. Correlation Heatmap
Compute Pearson correlation matrix on numeric features.


In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Matrix of Numeric Features')
plt.show()


## 8. Outlier Detection using IQR Method
Calculate number of outliers in `BMI` and `Avg_Glucose_Level`.


In [ ]:
for col in ['Avg_Glucose_Level', 'BMI']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f'{col}: lower bound = {lower_bound:.2f}, upper bound = {upper_bound:.2f}')
    print(f'  Total Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)')
